In [1]:
# ==========================================
# 0. INSTALL — Qwen LoRA retriever + reranker deps
# Fresh Colab A100 runtime: Python 3.12 + torch 2.10.0+cu128
# ==========================================

import sys
import subprocess

INSTALLATION_OK = False

def run(cmd):
    print(f"\n$ {cmd}", flush=True)
    result = subprocess.run(cmd, shell=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")

# Keep Colab's existing torch. Do NOT reinstall torch.
run(f"""{sys.executable} -m pip install -U \
"transformers>=4.51.0" \
"sentence-transformers>=5.0.0" \
"datasets>=2.19.0" \
"accelerate>=0.30.0" \
"peft>=0.12.0" \
safetensors tqdm scikit-learn packaging ninja""")

# PEFT compatibility fix for torch 2.10.0+cu128
run(f"{sys.executable} -m pip install -U torchao --index-url https://download.pytorch.org/whl/cu128")

# Prebuilt FlashAttention wheel for torch 2.10 + Python 3.12 + CUDA 12.8
FLASH_ATTN_WHEEL = "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
run(f'{sys.executable} -m pip install -U "{FLASH_ATTN_WHEEL}"')

# Verification
import torch
import transformers
import flash_attn
import torchao
from transformers.utils import is_flash_attn_2_available

print("\n===== VERIFICATION =====")
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA used by torch:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else None)
print("transformers:", transformers.__version__)
print("torchao:", torchao.__version__)
print("flash_attn:", getattr(flash_attn, "__version__", "unknown"))
print("FA2 available to Transformers:", is_flash_attn_2_available())

assert torch.cuda.is_available(), "CUDA is not available"
assert torch.cuda.is_bf16_supported(), "BF16 is not supported"
assert is_flash_attn_2_available(), "FlashAttention 2 is not available to Transformers"

INSTALLATION_OK = True
print("\n✅ INSTALLATION_OK = True")


$ /usr/bin/python3 -m pip install -U "transformers>=4.51.0" "sentence-transformers>=5.0.0" "datasets>=2.19.0" "accelerate>=0.30.0" "peft>=0.12.0" safetensors tqdm scikit-learn packaging ninja

$ /usr/bin/python3 -m pip install -U torchao --index-url https://download.pytorch.org/whl/cu128

$ /usr/bin/python3 -m pip install -U "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"



===== VERIFICATION =====
torch: 2.10.0+cu128
CUDA available: True
CUDA used by torch: 12.8
GPU: NVIDIA A100-SXM4-40GB
BF16 supported: True
transformers: 5.7.0
torchao: 0.17.0+cu128
flash_attn: 2.8.3
FA2 available to Transformers: True

✅ INSTALLATION_OK = True


In [2]:
assert INSTALLATION_OK is True, "Install cell failed. Do not start reranker pipeline."

In [3]:
# ==========================================
# CT26 TASK 1 — STAGE 2 RERANKER FINETUNING
# Uses Qwen3-8B LoRA retriever ONLY to build top-10 candidates,
# then unloads Qwen before training BGE reranker.
# ==========================================

assert INSTALLATION_OK is True, "Install cell failed. Do not start pipeline."

# ==========================================
# 1. IMPORTS
# ==========================================
import os
import gc
import json
import gzip
import math
import random
import shutil
from datetime import datetime

import numpy as np
import torch
import torch.nn.functional as F

from datasets import load_dataset
from tqdm.auto import tqdm
from torch.utils.data import Dataset as TorchDataset, DataLoader

from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

# ==========================================
# 2. GOOGLE DRIVE + PATHS
# ==========================================
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_ROOT = "/content/drive/MyDrive"
    except Exception as e:
        print("Drive mount failed:", repr(e))
        DRIVE_ROOT = "."
else:
    DRIVE_ROOT = "."

# Qwen LoRA Stage-1 retriever paths
STAGE1_ROOT = os.path.join(DRIVE_ROOT, "ct26_qwen3_embedding_8b_lora")
BEST_RETRIEVER_DIR = os.path.join(STAGE1_ROOT, "best_qwen3_8b_lora_sentence_transformer")
RETRIEVER_ARTIFACTS_DIR = os.path.join(STAGE1_ROOT, "retrieval_artifacts")

# New stage-2 folder
STAGE2_ROOT = os.path.join(DRIVE_ROOT, "ct26_stage2_reranker_listwise_top10_qwen3_lora_bge_m3_mrr5")
CHECKPOINT_DIR = os.path.join(STAGE2_ROOT, "trainer_checkpoints")
BEST_RERANKER_DIR = os.path.join(STAGE2_ROOT, "best_reranker_model")
CACHE_DIR = os.path.join(STAGE2_ROOT, "cache")
EVAL_DIR = os.path.join(STAGE2_ROOT, "eval")

RESET_STAGE2_ROOT = True
BACKUP_EXISTING_STAGE2_ROOT = True

if RESET_STAGE2_ROOT and os.path.isdir(STAGE2_ROOT):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    if BACKUP_EXISTING_STAGE2_ROOT:
        backup_dir = STAGE2_ROOT + f"_BACKUP_{timestamp}"
        print(f"Existing STAGE2_ROOT found. Moving to backup:\n{backup_dir}")
        shutil.move(STAGE2_ROOT, backup_dir)
    else:
        print(f"Existing STAGE2_ROOT found. Deleting:\n{STAGE2_ROOT}")
        shutil.rmtree(STAGE2_ROOT)

for p in [STAGE2_ROOT, CHECKPOINT_DIR, BEST_RERANKER_DIR, CACHE_DIR, EVAL_DIR]:
    os.makedirs(p, exist_ok=True)

print("STAGE1_ROOT:", STAGE1_ROOT)
print("STAGE2_ROOT:", STAGE2_ROOT)

# ==========================================
# 3. CONFIG
# ==========================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

HF_TOKEN = os.environ.get("HF_TOKEN", None)
DATASET_NAME = "sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims"
LANGUAGES = ["en", "fr", "de"]

TASK_INSTRUCTION = (
    "Given a scientific web claim, retrieve the title and abstract of the "
    "scientific publication that is the source or best evidence for the claim."
)

def get_detailed_instruct(task_description, query):
    return f"Instruct: {task_description}\nQuery:{query}"

# Reranker
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
RERANKER_MAX_LENGTH = 512
TRAIN_GROUP_BATCH_SIZE = 32
PAIR_EVAL_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 1

MAX_EPOCHS = 3
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 2
MAX_GRAD_NORM = 1.0
LOGGING_STEPS = 100

# Strict top-10 apples-to-apples setup
TOPK = 10
TRAIN_RETRIEVAL_TOPK = TOPK
DEV_RERANK_K = TOPK
LISTWISE_GROUP_SIZE = TOPK
assert LISTWISE_GROUP_SIZE == 10

# Qwen retrieval batch size
QUERY_BATCH_SIZE_RETRIEVER = 32

# Misc
DELETE_TRAINER_CHECKPOINTS_AFTER_EXPORT = True
FORCE_REBUILD_TRAIN_GROUPS = True
FORCE_REBUILD_DEV_CANDIDATES = True
LIMIT_TRAIN_PER_LANG = None

FUSION_ALPHAS = [round(x, 1) for x in np.linspace(0.0, 1.0, 11)]

TRAIN_GROUPS_GZ = os.path.join(CACHE_DIR, f"train_listwise_groups_top{TOPK}.json.gz")
DEV_CANDIDATES_GZ = os.path.join(CACHE_DIR, f"dev_candidates_top{DEV_RERANK_K}.json.gz")
SCORED_DEV_GZ = os.path.join(EVAL_DIR, f"dev_scored_candidates_top{DEV_RERANK_K}_listwise_logits.json.gz")
META_PATH = os.path.join(STAGE2_ROOT, "meta.json")

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16

# ==========================================
# 4. HELPERS
# ==========================================
def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

def print_gpu_memory(label):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"[GPU MEM] {label}: allocated={allocated:.2f}GB reserved={reserved:.2f}GB")

def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json_gz(path, obj):
    with gzip.open(path, "wt", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)

def load_json_gz(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return json.load(f)

def normalize_ws(text):
    return " ".join(str(text).split())

def load_ct26_split(lang, split):
    kwargs = {}
    if HF_TOKEN:
        kwargs["token"] = HF_TOKEN
    return load_dataset(DATASET_NAME, lang, split=split, **kwargs)

def zscore(x):
    x = np.asarray(x, dtype=np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)

def compute_metrics_from_ranks(ranks):
    ranks = np.asarray(ranks)
    return {
        "MRR@1": float(np.mean([1.0 / r if r <= 1 else 0.0 for r in ranks])),
        "MRR@5": float(np.mean([1.0 / r if r <= 5 else 0.0 for r in ranks])),
        "MRR@10": float(np.mean([1.0 / r if r <= 10 else 0.0 for r in ranks])),
        "Recall@5": float((ranks <= 5).mean()),
        "Recall@10": float((ranks <= 10).mean()),
    }

def get_topk_from_scores(scores, k):
    k = min(k, scores.shape[1])
    part = np.argpartition(-scores, kth=k - 1, axis=1)[:, :k]
    part_scores = np.take_along_axis(scores, part, axis=1)
    order = np.argsort(-part_scores, axis=1)
    return np.take_along_axis(part, order, axis=1)

def batched_retrieve_topk(query_texts, retriever, doc_matrix, topk, batch_size=32):
    all_topk_idx = []
    all_topk_scores = []

    for start in tqdm(range(0, len(query_texts), batch_size), desc=f"Qwen retriever top-{topk}"):
        batch_queries = query_texts[start:start + batch_size]

        with torch.inference_mode():
            q_emb = retriever.encode(
                batch_queries,
                batch_size=batch_size,
                convert_to_numpy=True,
                normalize_embeddings=True,
                show_progress_bar=False,
            ).astype(np.float32)

        scores = q_emb @ doc_matrix.T
        batch_topk_idx = get_topk_from_scores(scores, topk)
        batch_topk_scores = np.take_along_axis(scores, batch_topk_idx, axis=1)

        all_topk_idx.extend(batch_topk_idx.tolist())
        all_topk_scores.extend(batch_topk_scores.astype(np.float32).tolist())

        del q_emb, scores, batch_topk_idx, batch_topk_scores
        cleanup_cuda()

    return all_topk_idx, all_topk_scores

def print_multilingual_metrics(title, metrics):
    print(f"\n===== {title} =====")
    for lang in LANGUAGES:
        print(f"--- {lang.upper()} ---")
        print(f"  MRR@1     : {metrics[f'{lang}_mrr@1']:.4f}")
        print(f"  MRR@5     : {metrics[f'{lang}_mrr@5']:.4f}")
        print(f"  MRR@10    : {metrics[f'{lang}_mrr@10']:.4f}")
        print(f"  Recall@5  : {metrics[f'{lang}_recall@5']:.4f}")
        print(f"  Recall@10 : {metrics[f'{lang}_recall@10']:.4f}")
    print("===== MULTILINGUAL AVG =====")
    print(f"  MRR@1     : {metrics['multilingual_avg_mrr@1']:.4f}")
    print(f"  MRR@5     : {metrics['multilingual_avg_mrr@5']:.4f}")
    print(f"  MRR@10    : {metrics['multilingual_avg_mrr@10']:.4f}")
    print(f"  Recall@5  : {metrics['multilingual_avg_recall@5']:.4f}")
    print(f"  Recall@10 : {metrics['multilingual_avg_recall@10']:.4f}")

# ==========================================
# 5. CANDIDATE BUILDING
# ==========================================
def build_train_listwise_groups(
    combined_train,
    train_topk_idx,
    doc_ids_list,
    doc_raw,
    group_size=10,
    seed=42,
):
    groups = []
    skipped = 0

    for i, item in enumerate(tqdm(combined_train, desc="Building listwise train groups")):
        q_raw = normalize_ws(item["text"])
        gold_id = str(item["pubkey"])
        lang = item["lang"]

        if gold_id not in doc_raw:
            skipped += 1
            continue

        retrieved_ids = [doc_ids_list[idx] for idx in train_topk_idx[i]]
        wrong_ids = [d for d in retrieved_ids if d != gold_id]

        if len(wrong_ids) < group_size - 1:
            skipped += 1
            continue

        if gold_id in retrieved_ids:
            group_ids = retrieved_ids[:group_size]
        else:
            group_ids = [gold_id] + wrong_ids[:group_size - 1]

        group_ids = group_ids[:group_size]
        if gold_id not in group_ids:
            group_ids[-1] = gold_id

        rng = np.random.default_rng(seed + i)
        perm = rng.permutation(group_size)
        shuffled_ids = [group_ids[j] for j in perm]
        positive_index = shuffled_ids.index(gold_id)

        groups.append({
            "query": q_raw,  # raw query for reranker
            "candidate_ids": shuffled_ids,
            "candidate_texts": [doc_raw[did] for did in shuffled_ids],
            "positive_index": positive_index,
            "gold_id": gold_id,
            "lang": lang,
        })

    print(f"Skipped train items: {skipped}")
    return groups

def build_dev_candidates(
    dev_by_lang_raw,
    retriever,
    doc_matrix,
    doc_ids_list,
    doc_raw,
    topk=10,
    batch_size=32,
):
    full_dev_samples_by_lang = {}

    for lang in LANGUAGES:
        dev_split = dev_by_lang_raw[lang]
        queries_raw = [normalize_ws(x["text"]) for x in dev_split]
        queries_for_retriever = [get_detailed_instruct(TASK_INSTRUCTION, q) for q in queries_raw]
        gold_ids = [str(x["pubkey"]) for x in dev_split]

        topk_idx, topk_scores = batched_retrieve_topk(
            query_texts=queries_for_retriever,
            retriever=retriever,
            doc_matrix=doc_matrix,
            topk=topk,
            batch_size=batch_size,
        )

        samples = []
        for i, gold_id in enumerate(tqdm(gold_ids, desc=f"Building dev samples {lang.upper()}")):
            candidate_ids = [doc_ids_list[idx] for idx in topk_idx[i]]
            candidate_texts = [doc_raw[did] for did in candidate_ids]

            samples.append({
                "query": queries_raw[i],  # raw query for reranker
                "gold_id": gold_id,
                "candidate_ids": candidate_ids,
                "candidate_texts": candidate_texts,
                "dense_scores": topk_scores[i],
            })

        full_dev_samples_by_lang[lang] = samples

    return full_dev_samples_by_lang

# ==========================================
# 6. DATASET + COLLATOR
# ==========================================
class ListwiseGroupDataset(TorchDataset):
    def __init__(self, groups):
        self.groups = groups

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        return self.groups[idx]

class ListwiseCollator:
    def __init__(self, tokenizer, max_length, group_size):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.group_size = group_size

    def __call__(self, batch):
        queries = []
        docs = []
        positive_indices = []

        for item in batch:
            assert len(item["candidate_texts"]) == self.group_size
            queries.extend([item["query"]] * self.group_size)
            docs.extend(item["candidate_texts"])
            positive_indices.append(item["positive_index"])

        features = self.tokenizer(
            queries,
            docs,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        features["positive_indices"] = torch.tensor(positive_indices, dtype=torch.long)
        features["num_groups"] = len(batch)
        return features

# ==========================================
# 7. RERANKER SCORING / EVAL / FUSION
# ==========================================
def autotune_inference_batch_size(model, tokenizer, sample_pairs, max_length, device):
    candidates = [256, 192, 128, 96, 64, 48, 32, 24, 16, 8]
    was_training = model.training
    model.eval()

    try:
        for bs in candidates:
            try:
                probe = sample_pairs[:bs]
                q = [x[0] for x in probe]
                d = [x[1] for x in probe]

                features = tokenizer(
                    q,
                    d,
                    padding=True,
                    truncation=True,
                    max_length=max_length,
                    return_tensors="pt",
                )
                features = {k: v.to(device, non_blocking=True) for k, v in features.items()}

                cleanup_cuda()

                with torch.inference_mode():
                    with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=USE_BF16):
                        logits = model(**features).logits

                _ = logits.float().view(-1).cpu().numpy()
                print(f"Autotuned inference batch size: {bs}")
                return bs

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"Inference batch size {bs} OOM, trying smaller...")
                    cleanup_cuda()
                else:
                    raise

        raise RuntimeError("Could not find a working inference batch size.")

    finally:
        if was_training:
            model.train()

def score_flat_pairs_exact_fast(model, tokenizer, pairs, max_length, batch_size, device):
    n = len(pairs)
    if n == 0:
        return np.zeros((0,), dtype=np.float32)

    was_training = model.training
    model.eval()

    try:
        est_lengths = np.asarray([len(q.split()) + len(d.split()) for q, d in pairs], dtype=np.int32)
        order = np.argsort(est_lengths)
        sorted_pairs = [pairs[i] for i in order]

        sorted_scores = np.empty(n, dtype=np.float32)

        for start in tqdm(range(0, n, batch_size), desc="  pair batches", leave=False):
            batch_pairs = sorted_pairs[start:start + batch_size]
            q = [x[0] for x in batch_pairs]
            d = [x[1] for x in batch_pairs]

            features = tokenizer(
                q,
                d,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            features = {k: v.to(device, non_blocking=True) for k, v in features.items()}

            with torch.inference_mode():
                with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=USE_BF16):
                    logits = model(**features).logits

            batch_scores = logits.float().view(-1).detach().cpu().numpy().astype(np.float32)
            sorted_scores[start:start + len(batch_scores)] = batch_scores

        unsorted_scores = np.empty(n, dtype=np.float32)
        unsorted_scores[order] = sorted_scores
        return unsorted_scores

    finally:
        if was_training:
            model.train()

def evaluate_reranker_on_candidates(model, tokenizer, samples_by_lang, max_length, pair_batch_size, device, name="dev"):
    was_training = model.training
    model.eval()

    try:
        metrics = {}
        per_lang = {}

        print(f"\n===== RERANKER EVAL ({name}) =====")
        for lang in LANGUAGES:
            samples = samples_by_lang[lang]

            flat_pairs = []
            counts = []
            candidate_ids_all = []
            gold_ids_all = []

            for sample in samples:
                q = sample["query"]
                docs = sample["candidate_texts"]
                counts.append(len(docs))
                candidate_ids_all.append(sample["candidate_ids"])
                gold_ids_all.append(sample["gold_id"])
                flat_pairs.extend((q, d) for d in docs)

            flat_scores = score_flat_pairs_exact_fast(
                model=model,
                tokenizer=tokenizer,
                pairs=flat_pairs,
                max_length=max_length,
                batch_size=pair_batch_size,
                device=device,
            )

            ranks = []
            offset = 0
            for count, candidate_ids, gold_id in zip(counts, candidate_ids_all, gold_ids_all):
                ce_scores = flat_scores[offset:offset + count]
                order = np.argsort(-ce_scores)
                reranked_ids = [candidate_ids[i] for i in order]

                rank = reranked_ids.index(gold_id) + 1 if gold_id in reranked_ids else 10000
                ranks.append(rank)
                offset += count

            lang_scores = compute_metrics_from_ranks(ranks)
            per_lang[lang] = lang_scores

            metrics[f"{lang}_mrr@1"] = lang_scores["MRR@1"]
            metrics[f"{lang}_mrr@5"] = lang_scores["MRR@5"]
            metrics[f"{lang}_mrr@10"] = lang_scores["MRR@10"]
            metrics[f"{lang}_recall@5"] = lang_scores["Recall@5"]
            metrics[f"{lang}_recall@10"] = lang_scores["Recall@10"]

        metrics["multilingual_avg_mrr@1"] = float(np.mean([per_lang[l]["MRR@1"] for l in LANGUAGES]))
        metrics["multilingual_avg_mrr@5"] = float(np.mean([per_lang[l]["MRR@5"] for l in LANGUAGES]))
        metrics["multilingual_avg_mrr@10"] = float(np.mean([per_lang[l]["MRR@10"] for l in LANGUAGES]))
        metrics["multilingual_avg_recall@5"] = float(np.mean([per_lang[l]["Recall@5"] for l in LANGUAGES]))
        metrics["multilingual_avg_recall@10"] = float(np.mean([per_lang[l]["Recall@10"] for l in LANGUAGES]))

        print_multilingual_metrics(f"RERANKER EVAL ({name})", metrics)
        return metrics

    finally:
        if was_training:
            model.train()

def precompute_ce_scores_for_candidates(model, tokenizer, samples_by_lang, max_length, pair_batch_size, device):
    scored = {}

    print("\nPrecomputing reranker logits for alpha sweep...")
    for lang in LANGUAGES:
        samples = samples_by_lang[lang]

        flat_pairs = []
        counts = []

        for sample in samples:
            q = sample["query"]
            docs = sample["candidate_texts"]
            counts.append(len(docs))
            flat_pairs.extend((q, d) for d in docs)

        print(f"\nLanguage: {lang.upper()} | queries={len(samples)} | pairs={len(flat_pairs)}")

        flat_scores = score_flat_pairs_exact_fast(
            model=model,
            tokenizer=tokenizer,
            pairs=flat_pairs,
            max_length=max_length,
            batch_size=pair_batch_size,
            device=device,
        )

        scored_samples = []
        offset = 0
        for sample, count in zip(samples, counts):
            new_sample = dict(sample)
            new_sample["ce_scores"] = flat_scores[offset:offset + count].tolist()
            scored_samples.append(new_sample)
            offset += count

        scored[lang] = scored_samples

        partial_path = os.path.join(EVAL_DIR, f"dev_scored_candidates_top{DEV_RERANK_K}_{lang}_partial.json.gz")
        save_json_gz(partial_path, {lang: scored_samples})
        print(f"Saved partial scores to: {partial_path}")

    return scored

def evaluate_alpha_from_cached(scored_samples_by_lang, alpha, name="alpha-sweep"):
    metrics = {}
    per_lang = {}

    print(f"\n===== EVAL ({name}, alpha={alpha:.1f}) =====")
    for lang in LANGUAGES:
        samples = scored_samples_by_lang[lang]
        ranks = []

        for sample in tqdm(samples, desc=f"Evaluating {lang.upper()} @ alpha={alpha:.1f}", leave=False):
            gold_id = sample["gold_id"]
            candidate_ids = sample["candidate_ids"]

            dense_scores = np.asarray(sample["dense_scores"], dtype=np.float32)
            ce_scores = np.asarray(sample["ce_scores"], dtype=np.float32)

            final_scores = alpha * zscore(dense_scores) + (1.0 - alpha) * zscore(ce_scores)

            order = np.argsort(-final_scores)
            reranked_ids = [candidate_ids[i] for i in order]

            rank = reranked_ids.index(gold_id) + 1 if gold_id in reranked_ids else 10000
            ranks.append(rank)

        lang_scores = compute_metrics_from_ranks(ranks)
        per_lang[lang] = lang_scores

        metrics[f"{lang}_mrr@1"] = lang_scores["MRR@1"]
        metrics[f"{lang}_mrr@5"] = lang_scores["MRR@5"]
        metrics[f"{lang}_mrr@10"] = lang_scores["MRR@10"]
        metrics[f"{lang}_recall@5"] = lang_scores["Recall@5"]
        metrics[f"{lang}_recall@10"] = lang_scores["Recall@10"]

    metrics["multilingual_avg_mrr@1"] = float(np.mean([per_lang[l]["MRR@1"] for l in LANGUAGES]))
    metrics["multilingual_avg_mrr@5"] = float(np.mean([per_lang[l]["MRR@5"] for l in LANGUAGES]))
    metrics["multilingual_avg_mrr@10"] = float(np.mean([per_lang[l]["MRR@10"] for l in LANGUAGES]))
    metrics["multilingual_avg_recall@5"] = float(np.mean([per_lang[l]["Recall@5"] for l in LANGUAGES]))
    metrics["multilingual_avg_recall@10"] = float(np.mean([per_lang[l]["Recall@10"] for l in LANGUAGES]))

    print_multilingual_metrics(f"EVAL ({name}, alpha={alpha:.1f})", metrics)
    return metrics

# ==========================================
# 8. LOAD STATIC RETRIEVER ARTIFACTS
# ==========================================
print("\nLoading Qwen stage-1 artifacts...")
assert os.path.isdir(BEST_RETRIEVER_DIR), f"Missing retriever model dir: {BEST_RETRIEVER_DIR}"
assert os.path.isdir(RETRIEVER_ARTIFACTS_DIR), f"Missing artifacts dir: {RETRIEVER_ARTIFACTS_DIR}"

print("Loading Qwen doc embeddings from disk...")
doc_matrix = np.load(os.path.join(RETRIEVER_ARTIFACTS_DIR, "qwen3_8b_lora_doc_embeddings.npy")).astype(np.float32)

doc_ids_list = load_json(os.path.join(RETRIEVER_ARTIFACTS_DIR, "doc_ids_list.json"))
doc_raw = load_json_gz(os.path.join(RETRIEVER_ARTIFACTS_DIR, "doc_raw.json.gz"))

doc_ids_list = [str(x) for x in doc_ids_list]
doc_raw = {str(k): v for k, v in doc_raw.items()}
docid_to_idx = {did: idx for idx, did in enumerate(doc_ids_list)}

print(f"Loaded {len(doc_ids_list)} docs")
print(f"Doc embedding matrix shape: {doc_matrix.shape}")

# ==========================================
# 9. LOAD TRAIN + DEV SPLITS
# ==========================================
print("\nLoading train/dev splits...")
combined_train = []
dev_by_lang_raw = {}

for lang in LANGUAGES:
    train_split = load_ct26_split(lang, "train")
    dev_split = load_ct26_split(lang, "dev")

    if LIMIT_TRAIN_PER_LANG is not None:
        train_split = train_split.select(range(min(LIMIT_TRAIN_PER_LANG, len(train_split))))

    for item in train_split:
        row = dict(item)
        row["lang"] = lang
        combined_train.append(row)

    dev_by_lang_raw[lang] = dev_split

print(f"Total train queries: {len(combined_train)}")
for lang in LANGUAGES:
    print(f"Dev {lang}: {len(dev_by_lang_raw[lang])}")

# ==========================================
# 10. LOAD QWEN RETRIEVER ONLY FOR CANDIDATE GENERATION
# ==========================================
need_qwen_retriever = (
    FORCE_REBUILD_TRAIN_GROUPS
    or FORCE_REBUILD_DEV_CANDIDATES
    or not os.path.isfile(TRAIN_GROUPS_GZ)
    or not os.path.isfile(DEV_CANDIDATES_GZ)
)

retriever = None

if need_qwen_retriever:
    print("\nLoading Qwen LoRA retriever with FlashAttention 2 for candidate generation only...")
    print_gpu_memory("before Qwen load")

    try:
        retriever = SentenceTransformer(
            BEST_RETRIEVER_DIR,
            model_kwargs={
                "attn_implementation": "flash_attention_2",
                "dtype": torch.bfloat16,
                "device_map": {"": 0},
            },
            processor_kwargs={
                "padding_side": "left",
            },
        )
    except TypeError:
        retriever = SentenceTransformer(
            BEST_RETRIEVER_DIR,
            model_kwargs={
                "attn_implementation": "flash_attention_2",
                "torch_dtype": torch.bfloat16,
                "device_map": {"": 0},
            },
            tokenizer_kwargs={
                "padding_side": "left",
            },
        )

    retriever.max_seq_length = 512
    retriever.tokenizer.padding_side = "left"
    retriever.eval()

    print_gpu_memory("after Qwen load")
else:
    print("\nCandidate caches exist and FORCE flags are False, so Qwen retriever will not be loaded.")

# ==========================================
# 11. BUILD OR LOAD TRAIN GROUPS
# ==========================================
print("\nPreparing listwise train groups...")

if os.path.isfile(TRAIN_GROUPS_GZ) and not FORCE_REBUILD_TRAIN_GROUPS:
    print("Loading cached train groups...")
    train_groups = load_json_gz(TRAIN_GROUPS_GZ)
else:
    assert retriever is not None, "Qwen retriever is required to build train groups."

    train_queries_for_retriever = []
    filtered_train_rows = []

    for item in combined_train:
        q_raw = normalize_ws(item["text"])
        gold_id = str(item["pubkey"])

        if gold_id not in doc_raw:
            continue

        train_queries_for_retriever.append(get_detailed_instruct(TASK_INSTRUCTION, q_raw))
        filtered_train_rows.append(item)

    train_topk_idx, _ = batched_retrieve_topk(
        query_texts=train_queries_for_retriever,
        retriever=retriever,
        doc_matrix=doc_matrix,
        topk=TRAIN_RETRIEVAL_TOPK,
        batch_size=QUERY_BATCH_SIZE_RETRIEVER,
    )

    train_groups = build_train_listwise_groups(
        combined_train=filtered_train_rows,
        train_topk_idx=train_topk_idx,
        doc_ids_list=doc_ids_list,
        doc_raw=doc_raw,
        group_size=LISTWISE_GROUP_SIZE,
        seed=SEED,
    )

    save_json_gz(TRAIN_GROUPS_GZ, train_groups)
    print(f"Saved train groups to: {TRAIN_GROUPS_GZ}")

print(f"Train groups: {len(train_groups)}")

# ==========================================
# 12. BUILD OR LOAD DEV CANDIDATES
# ==========================================
print("\nPreparing dev top-10 candidate sets...")

if os.path.isfile(DEV_CANDIDATES_GZ) and not FORCE_REBUILD_DEV_CANDIDATES:
    print("Loading cached dev candidates...")
    full_dev_samples_by_lang = load_json_gz(DEV_CANDIDATES_GZ)
else:
    assert retriever is not None, "Qwen retriever is required to build dev candidates."

    full_dev_samples_by_lang = build_dev_candidates(
        dev_by_lang_raw=dev_by_lang_raw,
        retriever=retriever,
        doc_matrix=doc_matrix,
        doc_ids_list=doc_ids_list,
        doc_raw=doc_raw,
        topk=DEV_RERANK_K,
        batch_size=QUERY_BATCH_SIZE_RETRIEVER,
    )

    save_json_gz(DEV_CANDIDATES_GZ, full_dev_samples_by_lang)
    print(f"Saved dev candidates to: {DEV_CANDIDATES_GZ}")

for lang in LANGUAGES:
    print(f"Dev candidate pools {lang}: {len(full_dev_samples_by_lang[lang])}")

# ==========================================
# 13. UNLOAD QWEN BEFORE LOADING RERANKER
# ==========================================
print("\nUnloading Qwen retriever before reranker training...")
print_gpu_memory("before Qwen unload")

if retriever is not None:
    try:
        retriever.cpu()
    except Exception:
        pass

    del retriever
    retriever = None

cleanup_cuda()
print_gpu_memory("after Qwen unload")

# doc_matrix is CPU numpy, candidate caches contain retriever scores.
# No Qwen model is needed from this point onward.

# ==========================================
# 14. LOAD RERANKER TOKENIZER + MODEL
# ==========================================
print("\nLoading tokenizer and reranker model...")

tokenizer = AutoTokenizer.from_pretrained(
    RERANKER_MODEL_NAME,
    use_fast=True,
    trust_remote_code=True,
)

model = AutoModelForSequenceClassification.from_pretrained(
    RERANKER_MODEL_NAME,
    trust_remote_code=True,
)

try:
    model.gradient_checkpointing_enable()
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False
    print("Enabled gradient checkpointing.")
except Exception as e:
    print(f"Could not enable gradient checkpointing: {e}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("CUDA available:", torch.cuda.is_available())
print("Using device:", device)
print("BF16:", USE_BF16, "| FP16:", USE_FP16)
print_gpu_memory("after reranker load")

# ==========================================
# 15. DATALOADER
# ==========================================
train_dataset = ListwiseGroupDataset(train_groups)

collator = ListwiseCollator(
    tokenizer=tokenizer,
    max_length=RERANKER_MAX_LENGTH,
    group_size=LISTWISE_GROUP_SIZE,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_GROUP_BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

# ==========================================
# 16. OPTIMIZER + SCHEDULER
# ==========================================
num_update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
max_train_steps = num_update_steps_per_epoch * MAX_EPOCHS
warmup_steps = int(max_train_steps * WARMUP_RATIO)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=max_train_steps,
)

scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)

# ==========================================
# 17. PRE-TRAIN EVAL
# ==========================================
print("\nPre-training reranker eval...")

pretrain_metrics = evaluate_reranker_on_candidates(
    model=model,
    tokenizer=tokenizer,
    samples_by_lang=full_dev_samples_by_lang,
    max_length=RERANKER_MAX_LENGTH,
    pair_batch_size=PAIR_EVAL_BATCH_SIZE,
    device=device,
    name="dev-pretrain",
)

# ==========================================
# 18. TRAIN LOOP
# ==========================================
print("\nStarting listwise reranker training...")

best_metric = -1.0
best_epoch = None
best_checkpoint_dir = None
epochs_without_improvement = 0
history = []

if os.path.isdir(CHECKPOINT_DIR):
    shutil.rmtree(CHECKPOINT_DIR)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

global_step = 0
optimizer.zero_grad(set_to_none=True)

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_loss_sum = 0.0
    epoch_loss_count = 0

    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{MAX_EPOCHS}")

    for step, batch in enumerate(progress, start=1):
        positive_indices = batch.pop("positive_indices").to(device)
        num_groups = batch.pop("num_groups")
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=USE_BF16):
            outputs = model(**batch)
            logits = outputs.logits.view(num_groups, LISTWISE_GROUP_SIZE)
            loss = F.cross_entropy(logits, positive_indices)

        loss_for_logging = float(loss.detach().cpu().item())
        epoch_loss_sum += loss_for_logging
        epoch_loss_count += 1

        loss = loss / GRAD_ACCUM_STEPS

        if USE_FP16:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        should_step = (step % GRAD_ACCUM_STEPS == 0) or (step == len(train_loader))

        if should_step:
            if USE_FP16:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()

            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        if step % LOGGING_STEPS == 0 or step == len(train_loader):
            avg_so_far = epoch_loss_sum / max(epoch_loss_count, 1)
            progress.set_postfix(loss=f"{avg_so_far:.4f}")

    train_loss_avg = epoch_loss_sum / max(epoch_loss_count, 1)
    print(f"\nEpoch {epoch} train loss: {train_loss_avg:.6f}")

    dev_metrics = evaluate_reranker_on_candidates(
        model=model,
        tokenizer=tokenizer,
        samples_by_lang=full_dev_samples_by_lang,
        max_length=RERANKER_MAX_LENGTH,
        pair_batch_size=PAIR_EVAL_BATCH_SIZE,
        device=device,
        name=f"dev-epoch{epoch}",
    )

    current_metric = dev_metrics["multilingual_avg_mrr@5"]

    history.append({
        "epoch": epoch,
        "train_loss": train_loss_avg,
        **dev_metrics,
    })

    checkpoint_dir = os.path.join(CHECKPOINT_DIR, f"checkpoint-epoch{epoch}")
    os.makedirs(checkpoint_dir, exist_ok=True)
    model.save_pretrained(checkpoint_dir)
    tokenizer.save_pretrained(checkpoint_dir)

    if current_metric > best_metric:
        best_metric = current_metric
        best_epoch = epoch
        best_checkpoint_dir = checkpoint_dir
        epochs_without_improvement = 0

        if os.path.isdir(BEST_RERANKER_DIR):
            shutil.rmtree(BEST_RERANKER_DIR)

        model.save_pretrained(BEST_RERANKER_DIR)
        tokenizer.save_pretrained(BEST_RERANKER_DIR)

        print(f"New best checkpoint at epoch {epoch}: multilingual_avg_mrr@5 = {best_metric:.4f}")

    else:
        epochs_without_improvement += 1
        print(f"No MRR@5 improvement. Patience: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping triggered.")
        break

save_json(os.path.join(EVAL_DIR, "training_history.json"), history)

print("\nTraining complete.")
print(f"Best epoch: {best_epoch}")
print(f"Best multilingual_avg_mrr@5: {best_metric:.4f}")
print(f"Best checkpoint dir: {best_checkpoint_dir}")
print(f"Best reranker exported to: {BEST_RERANKER_DIR}")

if DELETE_TRAINER_CHECKPOINTS_AFTER_EXPORT:
    print("Deleting intermediate checkpoint folders...")
    for name in os.listdir(CHECKPOINT_DIR):
        path = os.path.join(CHECKPOINT_DIR, name)
        if os.path.isdir(path) and name.startswith("checkpoint-"):
            shutil.rmtree(path)

# ==========================================
# 19. RELOAD BEST RERANKER
# ==========================================
print("\nReloading best reranker...")

best_tokenizer = AutoTokenizer.from_pretrained(BEST_RERANKER_DIR, use_fast=True)
best_model = AutoModelForSequenceClassification.from_pretrained(BEST_RERANKER_DIR)
best_model.to(device)
best_model.eval()

# Free current model before using best model if different object
del model
cleanup_cuda()
model = best_model
tokenizer = best_tokenizer

# ==========================================
# 20. FINAL PURE RERANKER EVAL
# ==========================================
final_reranker_metrics = evaluate_reranker_on_candidates(
    model=model,
    tokenizer=tokenizer,
    samples_by_lang=full_dev_samples_by_lang,
    max_length=RERANKER_MAX_LENGTH,
    pair_batch_size=PAIR_EVAL_BATCH_SIZE,
    device=device,
    name="full-dev-pure-reranker",
)

save_json(
    os.path.join(EVAL_DIR, f"reranker_top{DEV_RERANK_K}_pure_metrics.json"),
    final_reranker_metrics,
)

# ==========================================
# 21. AUTOTUNE INFERENCE BATCH SIZE
# ==========================================
probe_pairs = []

for lang in LANGUAGES:
    if len(full_dev_samples_by_lang[lang]) > 0:
        for sample in full_dev_samples_by_lang[lang][:8]:
            q = sample["query"]
            for d in sample["candidate_texts"]:
                probe_pairs.append((q, d))
        if len(probe_pairs) >= 64:
            break

INFER_BATCH_SIZE = autotune_inference_batch_size(
    model=model,
    tokenizer=tokenizer,
    sample_pairs=probe_pairs,
    max_length=RERANKER_MAX_LENGTH,
    device=device,
)

# ==========================================
# 22. PRECOMPUTE LOGITS FOR FUSION
# ==========================================
scored_full_dev_samples_by_lang = precompute_ce_scores_for_candidates(
    model=model,
    tokenizer=tokenizer,
    samples_by_lang=full_dev_samples_by_lang,
    max_length=RERANKER_MAX_LENGTH,
    pair_batch_size=INFER_BATCH_SIZE,
    device=device,
)

save_json_gz(SCORED_DEV_GZ, scored_full_dev_samples_by_lang)
print(f"\nSaved scored dev candidates to: {SCORED_DEV_GZ}")

# ==========================================
# 23. ALPHA SWEEP
# ==========================================
alpha_sweep_results = {}

for alpha in FUSION_ALPHAS:
    if alpha == 0.0:
        run_name = "full-dev-pure-reranker"
    elif alpha == 1.0:
        run_name = "full-dev-pure-retriever"
    else:
        run_name = f"full-dev-fusion-a{alpha:.1f}"

    results = evaluate_alpha_from_cached(
        scored_samples_by_lang=scored_full_dev_samples_by_lang,
        alpha=alpha,
        name=run_name,
    )

    alpha_key = f"{alpha:.1f}"
    alpha_sweep_results[alpha_key] = results

    save_json(
        os.path.join(EVAL_DIR, f"reranker_top{DEV_RERANK_K}_{run_name}_metrics.json"),
        results,
    )

alpha_summary = []

for alpha_str, metrics in alpha_sweep_results.items():
    alpha_summary.append({
        "alpha": float(alpha_str),
        "multilingual_avg_mrr@1": metrics["multilingual_avg_mrr@1"],
        "multilingual_avg_mrr@5": metrics["multilingual_avg_mrr@5"],
        "multilingual_avg_mrr@10": metrics["multilingual_avg_mrr@10"],
        "multilingual_avg_recall@5": metrics["multilingual_avg_recall@5"],
        "multilingual_avg_recall@10": metrics["multilingual_avg_recall@10"],
    })

alpha_summary = sorted(alpha_summary, key=lambda x: x["alpha"])

summary_path = os.path.join(EVAL_DIR, f"alpha_sweep_summary_top{DEV_RERANK_K}.json")
save_json(summary_path, alpha_summary)

best_by_mrr5 = max(alpha_summary, key=lambda x: x["multilingual_avg_mrr@5"])

print("\n===== BEST ALPHA BY MULTILINGUAL AVG MRR@5 =====")
for k, v in best_by_mrr5.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

print("\n===== ALPHA SWEEP LEADERBOARD =====")
for row in alpha_summary:
    print(
        f"alpha={row['alpha']:.1f} | "
        f"MRR@5={row['multilingual_avg_mrr@5']:.4f} | "
        f"MRR@10={row['multilingual_avg_mrr@10']:.4f} | "
        f"Recall@5={row['multilingual_avg_recall@5']:.4f}"
    )

# ==========================================
# 24. SAVE META
# ==========================================
meta = {
    "stage1_root": STAGE1_ROOT,
    "best_retriever_dir": BEST_RETRIEVER_DIR,
    "retriever_artifacts_dir": RETRIEVER_ARTIFACTS_DIR,
    "stage2_root": STAGE2_ROOT,
    "best_reranker_dir": BEST_RERANKER_DIR,
    "checkpoint_dir": CHECKPOINT_DIR,
    "cache_dir": CACHE_DIR,
    "eval_dir": EVAL_DIR,
    "dataset_name": DATASET_NAME,
    "languages": LANGUAGES,
    "task_instruction_for_qwen_retriever": TASK_INSTRUCTION,
    "reranker_model_name": RERANKER_MODEL_NAME,
    "reranker_max_length": RERANKER_MAX_LENGTH,
    "topk": TOPK,
    "train_retrieval_topk": TRAIN_RETRIEVAL_TOPK,
    "dev_rerank_k": DEV_RERANK_K,
    "listwise_group_size": LISTWISE_GROUP_SIZE,
    "train_group_batch_size": TRAIN_GROUP_BATCH_SIZE,
    "pair_eval_batch_size": PAIR_EVAL_BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "max_epochs": MAX_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "best_epoch": best_epoch,
    "best_metric_multilingual_avg_mrr@5": best_metric,
    "seed": SEED,
    "loss_type": "listwise_softmax_cross_entropy",
    "alpha_sweep_best_by_mrr5": best_by_mrr5,
    "important_note": "Qwen retriever was unloaded before reranker model was loaded.",
}

save_json(META_PATH, meta)

print("\nSaved files:")
print("Best reranker:", BEST_RERANKER_DIR)
print("Train groups cache:", TRAIN_GROUPS_GZ)
print("Dev candidates cache:", DEV_CANDIDATES_GZ)
print("Scored dev candidates:", SCORED_DEV_GZ)
print("Alpha summary:", summary_path)
print("Meta file:", META_PATH)

# ==========================================
# 25. CLEANUP
# ==========================================
gc.collect()
cleanup_cuda()

print("\nDone.")

Mounted at /content/drive
Existing STAGE2_ROOT found. Moving to backup:
/content/drive/MyDrive/ct26_stage2_reranker_listwise_top10_qwen3_lora_bge_m3_mrr5_BACKUP_20260504_143921
STAGE1_ROOT: /content/drive/MyDrive/ct26_qwen3_embedding_8b_lora
STAGE2_ROOT: /content/drive/MyDrive/ct26_stage2_reranker_listwise_top10_qwen3_lora_bge_m3_mrr5

Loading Qwen stage-1 artifacts...
Loading Qwen doc embeddings from disk...
Loaded 10000 docs
Doc embedding matrix shape: (10000, 4096)

Loading train/dev splits...


README.md: 0.00B [00:00, ?B/s]

en_train.json: 0.00B [00:00, ?B/s]

en_dev.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14977 [00:00<?, ? examples/s]

...


===== EVAL (full-dev-fusion-a0.9, alpha=0.9) =====
--- EN ---
  MRR@1     : 0.6438
  MRR@5     : 0.7126
  MRR@10    : 0.7177
  Recall@5  : 0.8138
  Recall@10 : 0.8510
--- FR ---
  MRR@1     : 0.6567
  MRR@5     : 0.7211
  MRR@10    : 0.7282
  Recall@5  : 0.8120
  Recall@10 : 0.8632
--- DE ---
  MRR@1     : 0.5751
  MRR@5     : 0.6277
  MRR@10    : 0.6394
  Recall@5  : 0.7073
  Recall@10 : 0.7902
===== MULTILINGUAL AVG =====
  MRR@1     : 0.6252
  MRR@5     : 0.6871
  MRR@10    : 0.6951
  Recall@5  : 0.7777
  Recall@10 : 0.8348

===== EVAL (full-dev-pure-retriever, alpha=1.0) =====


Evaluating EN @ alpha=1.0:   0%|          | 0/3905 [00:00<?, ?it/s]

Evaluating FR @ alpha=1.0:   0%|          | 0/702 [00:00<?, ?it/s]

Evaluating DE @ alpha=1.0:   0%|          | 0/386 [00:00<?, ?it/s]


===== EVAL (full-dev-pure-retriever, alpha=1.0) =====
--- EN ---
  MRR@1     : 0.6318
  MRR@5     : 0.7052
  MRR@10    : 0.7105
  Recall@5  : 0.8120
  Recall@10 : 0.8510
--- FR ---
  MRR@1     : 0.6425
  MRR@5     : 0.7101
  MRR@10    : 0.7179
  Recall@5  : 0.8063
  Recall@10 : 0.8632
--- DE ---
  MRR@1     : 0.5699
  MRR@5     : 0.6226
  MRR@10    : 0.6345
  Recall@5  : 0.7021
  Recall@10 : 0.7902
===== MULTILINGUAL AVG =====
  MRR@1     : 0.6147
  MRR@5     : 0.6793
  MRR@10    : 0.6876
  Recall@5  : 0.7735
  Recall@10 : 0.8348

===== BEST ALPHA BY MULTILINGUAL AVG MRR@5 =====
alpha: 0.5000
multilingual_avg_mrr@1: 0.6492
multilingual_avg_mrr@5: 0.7094
multilingual_avg_mrr@10: 0.7141
multilingual_avg_recall@5: 0.7996
multilingual_avg_recall@10: 0.8348

===== ALPHA SWEEP LEADERBOARD =====
alpha=0.0 | MRR@5=0.6764 | MRR@10=0.6840 | Recall@5=0.7788
alpha=0.1 | MRR@5=0.6834 | MRR@10=0.6905 | Recall@5=0.7833
alpha=0.2 | MRR@5=0.6922 | MRR@10=0.6987 | Recall@5=0.7886
alpha=0.3 | MRR@5=0.70